In [4]:
from itertools import permutations
import pandas as pd
import numpy as np
import itertools
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pd.set_option('display.max_columns', None)

In [ ]:
# ==========================================
# PRE-PROCESS THE DATASET
# ==========================================
# Loading our group project dataset from the data folder
df_raw = pd.read_csv('data/CombinedDatasetConservativeTWOSIDES.csv', sep='\t')

# Cleaning up the data: fixing capitalization differences and standardizing the messy severity names
df_interesting = (
    df_raw.copy()
    .assign(object=lambda d: d.object.str.lower().str.capitalize())
    .assign(precipitant=lambda d: d.precipitant.str.lower().str.capitalize())
    .assign(severity=lambda d: d.severity.replace(' ', np.nan))
    # Combining all the different database scoring systems (numbers and text) into standard Low/Medium/High
    .assign(severity=lambda d: d.severity.map({
        'Critical': 'High',
        'Significant': 'Medium',
        '1': 'Low',
        '2': 'Medium',
        '3': 'High',
        'Low': 'Low',       # Preserves existing 'Low' strings
        'Medium': 'Medium', # Preserves existing 'Medium' strings
        'High': 'High',     # Preserves existing 'High' strings
        'None': np.nan
    }))
    .assign(evidenceStatement=lambda d: d.evidenceStatement.fillna('No statement available.'))
    # Filtering out rows without severity so our demo only shows clear interaction flags
    .query('severity.notnull()') 
)

# Creating a sorted list of all unique drugs so our autocomplete dropdown works smoothly
unique_drugs = sorted(list(set(df_interesting['object'].unique()) | set(df_interesting['precipitant'].unique())))


# ==========================================
# DDI LOGIC 
# ==========================================
class DDI_Logic():
    
    def __init__(self, data: pd.DataFrame, patient_history: list):
        self.data = data
        # Standardizing case instantly so typos don't break our lookups later
        self.patient_history = [str(d).strip().lower().capitalize() for d in patient_history]
        self.precipitant = [] # This list will hold the new drugs we type in
        
    def find_DDI(self, all_drug_permutations: list):
        result = []
        
        for obj, precip in all_drug_permutations:
            # Query our processed dataframe for this specific pair direction
            df_DDI = self.data.query('object == @obj and precipitant == @precip')
            
            # If we found a match, grab the details safely without crashing on empty df cells
            if not df_DDI.empty:
                row = df_DDI.iloc[0] 
                result_dict = {
                    'combination': [obj, precip],
                    'DDI_found': True,
                    'severity': row['severity'] if pd.notna(row['severity']) else 'Unknown', 
                    'evidenceStatement': row['evidenceStatement'] if pd.notna(row['evidenceStatement']) else 'No entry statement.'
                }
            else:
                # Fallback dictionary if there's no entry found
                result_dict = {
                    'combination': [obj, precip],
                    'DDI_found': False,
                    'severity': 'None', 
                    'evidenceStatement': 'No interaction found in database.'
                }
            
            result.append(result_dict)
        return result
    
    def generate_permutations(self, drug_list) -> list:
        # Order matters for drug pairs (Victim vs Perpetrator), so we use permutations of size 2
        return list(itertools.permutations(drug_list, 2))
            
    def check_general_clashes(self) -> list:
        # Combine all current meds (past history + new additions) into one unified pool
        all_active_drugs = self.patient_history + self.precipitant
        # Generate every possible pairwise combination across the whole pool
        all_pairs = self.generate_permutations(all_active_drugs)
        return self.find_DDI(all_drug_permutations=all_pairs)
    
    def user_input_precipitant(self, new_drug: str) -> None:
        clean_drug = str(new_drug).strip().lower().capitalize()
        if clean_drug not in self.precipitant:
            self.precipitant.append(clean_drug)
        
    def show_drugs(self) -> None:
        print("Precipitants currently loaded:", self.precipitant)


# ==========================================
# IPYWIDGETS MINI "DASHBOARD" 
# ==========================================

# Initializing the class using the professor's sample history values
history_mock = ['Moricizine', 'Ziprasidone']
logic = DDI_Logic(data=df_interesting, patient_history=history_mock)

# Creating our interface layout widgets
title_html = widgets.HTML("<h2 style='color:#4338ca; font-family:sans-serif; margin-bottom: 5px;'>General DDI Screening Tool</h2><hr>")

# Combobox gives us a text input field with a built-in search dropdown filter
drug_search = widgets.Combobox(
    placeholder='Type to search medicine...',
    options=unique_drugs,
    description='New Med:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='400px')
)

add_btn = widgets.Button(description='Add Prescription', button_style='warning', icon='plus')
reset_btn = widgets.Button(description='Reset New Meds', button_style='danger', icon='trash')
input_row = widgets.HBox([drug_search, add_btn, reset_btn])

# Display slots for our text counters and table readouts
status_display = widgets.HTML()
output_display = widgets.Output()

def format_html_table(title, ddi_results):
    # Building a clean HTML table string to render nicely inside our notebook cell
    html = f"<h4 style='font-family:sans-serif; margin-top:15px; color:#1e293b;'>{title}</h4>"
    html += "<table style='width:100%; border-collapse:collapse; font-family:sans-serif; text-align:left; font-size:20px;'>"
    html += "<tr style='background-color:#f1f5f9; border-bottom:2px solid #cbd5e1;'><th style='padding:6px;'>Object</th><th style='padding:6px;'>Precipitant</th><th style='padding:6px;'>Severity</th><th style='padding:6px;'>Evidence / Clinical Statement</th></tr>"
    
    found_flags = 0
    for res in ddi_results:
        if res['DDI_found']:
            found_flags += 1
            sev = res['severity']
            
            # Color code the text blocks based on severity level
            color = "#64748b" 
            if sev == 'High': color = "#dc2626; font-weight:bold;" 
            elif sev == 'Medium': color = "#d97706; font-weight:bold;" 
            elif sev == 'Low': color = "#2563eb;" 
                
            html += f"<tr style='border-bottom:1px solid #e2e8f0;'>"
            html += f"<td style='padding:6px; font-weight:600;'>{res['combination'][0]}</td>"
            html += f"<td style='padding:6px; font-weight:600;'>{res['combination'][1]}</td>"
            html += f"<td style='padding:6px; color:{color};'>{sev}</td>"
            html += f"<td style='padding:6px; color:#475569;'>{res['evidenceStatement']}</td>"
            html += "</tr>"
            
    if found_flags == 0:
        html += "<tr><td colspan='4' style='padding:12px; text-align:center; color:#94a3b8; font-style:italic;'>No active clashes flagged across the current drug list.</td></tr>"
    
    html += "</table>"
    return html

def update_dashboard():
    # Update our status badge display so we see what lists are currently in memory
    status_html = f"""
    <div style='font-family:sans-serif; background-color:#f8fafc; padding:10px; border-radius:6px; border:1px solid #e2e8f0; margin-bottom:15px;'>
        <b>Patient History List:</b> {", ".join(logic.patient_history) if logic.patient_history else "None"}<br>
        <b>Newly Added List:</b> {", ".join(logic.precipitant) if logic.precipitant else "<span style='color:#94a3b8;'>Empty</span>"}
    </div>
    """
    status_display.value = status_html
    
    # Clear old tables and print the freshly calculated unified table
    with output_display:
        clear_output(wait=True)
        
        # Run our unified check loop across all meds instead of splitting them up
        all_clashes = logic.check_general_clashes()
        display(HTML(format_html_table("⚠️ Medication DDI Detected", all_clashes)))

# Defining widget button click actions
def on_add_clicked(b):
    drug = drug_search.value
    if drug in unique_drugs:
        logic.user_input_precipitant(drug)
        drug_search.value = '' # Empty out the search input box
        update_dashboard()

def on_reset_clicked(b):
    logic.precipitant = [] # Wipe out only the freshly added drugs
    drug_search.value = ''
    update_dashboard()

# Link button objects to our operational functions
add_btn.on_click(on_add_clicked)
reset_btn.on_click(on_reset_clicked)

# Render everything out into the cell view area
display(title_html, input_row, status_display, output_display)
update_dashboard()

HTML(value="<h2 style='color:#4338ca; font-family:sans-serif; margin-bottom: 5px;'>General DDI Screening Tool<…

HTML(value='')

Output()

In [13]:
df_interesting

,drug1,object,drug2,precipitant,certainty,contraindication,dateAnnotated,ddiPkEffect,ddiPkMechanism,effectConcept,homepage,label,numericVal,objectUri,pathway,precaution,precipUri,severity,uri,whoAnnotated,source,ddiType,evidence,evidenceSource,evidenceStatement,researchStatementLabel,researchStatement
24780,http://bio2rdf.org/drugbank:DB01029,Irbesartan,http://bio2rdf.org/drugbank:DB00594,Amiloride,None,None,None,None,None,None,None,AMILORIDE/IRBESARTAN [VA Drug Interaction],None,None,None,None,None,Medium,http://purl.bioontology.org/ontology/NDFRT/N00...,None,NDF-RT,None,None,None,None,None,None
24781,http://bio2rdf.org/drugbank:DB00585,Nizatidine,http://bio2rdf.org/drugbank:DB04868,Nilotinib,None,None,None,None,None,None,None,NILOTINIB/NIZATIDINE [VA Drug Interaction],None,None,None,None,None,Medium,http://purl.bioontology.org/ontology/NDFRT/N00...,None,NDF-RT,None,None,None,None,None,None
24782,http://bio2rdf.org/drugbank:DB00543,Amoxapine,http://bio2rdf.org/drugbank:DB00615,Rifabutin,None,None,None,None,None,None,None,AMOXAPINE/RIFABUTIN [VA Drug Interaction],None,None,None,None,None,Medium,http://purl.bioontology.org/ontology/NDFRT/N00...,None,NDF-RT,None,None,None,None,None,None
24783,http://bio2rdf.org/drugbank:DB00344,Protriptyline,http://bio2rdf.org/drugbank:DB00615,Rifabutin,None,None,None,None,None,None,None,PROTRIPTYLINE/RIFABUTIN [VA Drug Interaction],None,None,None,None,None,Medium,http://purl.bioontology.org/ontology/NDFRT/N00...,None,NDF-RT,None,None,None,None,None,None
24784,http://bio2rdf.org/drugbank:DB00680,Moricizine,http://bio2rdf.org/drugbank:DB00199,Erythromycin,None,None,None,None,None,None,None,ERYTHROMYCIN/MORICIZINE [VA Drug Interaction],None,None,None,None,None,High,http://purl.bioontology.org/ontology/NDFRT/N00...,None,NDF-RT,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157039,http://bio2rdf.org/drugbank:DB00578,Carbenicillin,http://bio2rdf.org/drugbank:DB00571,Propranolol,None,None,None,None,None,I,None,None,None,None,None,None,None,Low,None,None,OSCAR,None,,None,No statement available.,NaN,NaN
157043,http://bio2rdf.org/drugbank:DB01203,Nadolol,http://bio2rdf.org/drugbank:DB00310,Chlorthalidone,None,None,None,None,None,A,None,None,None,None,None,None,None,Low,None,None,OSCAR,None,P,None,No statement available.,NaN,NaN
157045,http://bio2rdf.org/drugbank:DB01021,Trichlormethiazide,http://bio2rdf.org/drugbank:DB00390,Digoxin,None,None,None,None,None,A,None,None,None,None,None,None,None,Medium,None,None,OSCAR,None,P,None,"Digoxin toxicity, if potassium",NaN,NaN
157047,http://bio2rdf.org/drugbank:DB06724,Calcium_carbonate,http://bio2rdf.org/drugbank:DB00963,Bromfenac,None,None,None,None,None,I,None,None,None,None,None,None,None,Low,None,None,OSCAR,None,,None,No statement available.,NaN,NaN
